# Optuna Study Analysis

Load a completed (or in-progress) study from its journal file and inspect trial results.

**Setup:** set `STUDY_NAME` and `STORAGE_DIR` to match your `conf/study/*.yaml` config.

In [ ]:
STUDY_NAME  = "ssl_emnist_digits_clip"
STORAGE_DIR = "../sweeps"        # relative to this notebook

In [2]:
import os
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)

journal_path = os.path.join(STORAGE_DIR, f"{STUDY_NAME}.journal")
lock_path = journal_path + ".lock"
storage = optuna.storages.JournalStorage(
    optuna.storages.journal.JournalFileBackend(
        journal_path,
        lock_obj=optuna.storages.journal.JournalFileOpenLock(lock_path),
    )
)

study = optuna.load_study(study_name=STUDY_NAME, storage=storage)

trials_df = study.trials_dataframe()
completed = trials_df[trials_df["state"] == "COMPLETE"]

# Detect single vs multi-objective
multi_obj = len(study.directions) > 1

print(f"Study:      {study.study_name}")
print(f"Directions: {[d.name for d in study.directions]}")
print(f"Trials:     {len(study.trials)} total, {len(completed)} completed")
if not multi_obj:
    print(f"Best value: {study.best_value:.6f}  (trial #{study.best_trial.number})")
else:
    print(f"Pareto front: {len(study.best_trials)} trials")

/data/thallybu/packages/envs/mlresearch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Study:      ssl_emnist_digits_conv
Directions: ['MAXIMIZE']
Trials:     408 total, 122 completed
Best value: 0.921000  (trial #339)


## Best trial parameters

In [3]:
if multi_obj:
    rows = []
    for t in study.best_trials:
        row = {"trial": t.number, **dict(zip([f"obj_{i}" for i in range(len(t.values))], t.values)), **t.params}
        rows.append(row)
    pd.DataFrame(rows).set_index("trial")
else:
    pd.Series(study.best_trial.params, name="value").to_frame()

## All completed trials

In [4]:
param_cols = [c for c in trials_df.columns if c.startswith("params_")]
value_cols = [c for c in trials_df.columns if c.startswith("value")]
display_cols = ["number"] + value_cols + ["state", "duration"] + param_cols
sort_col = value_cols[0] if value_cols else "number"
completed[[c for c in display_cols if c in completed.columns]].sort_values(sort_col)

,number,value,state,duration,params_model.init_args.assignment_mode,params_model.init_args.assignment_threshold,params_model.init_args.assignments_per_epoch,params_model.init_args.autoencoder_kwargs.conv_channels,params_model.init_args.dropout,params_model.init_args.learning_rate,params_model.init_args.n_components,params_model.init_args.prototype_anchor_weight,params_model.init_args.prototypes_per_class,params_model.init_args.prune_per_epoch,params_model.init_args.prune_threshold,params_model.init_args.pruning_mode,params_model.init_args.umap_loss_weight,params_model.init_args.warmup_epochs
16,16,0.113500,COMPLETE,0 days 00:07:40.815530,0,0.549828,1,2,0.049436,0.000112,4,0.162528,2,0,1.336828,1,0.140640,5
31,31,0.416792,COMPLETE,0 days 00:11:45.404552,1,0.892798,3,3,0.105361,0.004506,19,0.094106,2,1,1.482661,0,0.021080,4
34,34,0.435792,COMPLETE,0 days 00:13:06.890466,1,0.542272,3,4,0.101566,0.001660,24,0.113944,2,1,1.493218,0,0.037274,3
10,10,0.473208,COMPLETE,0 days 00:21:51.991986,0,0.468541,4,4,0.053988,0.000479,7,0.245296,1,2,1.412726,0,0.150709,0
25,25,0.474458,COMPLETE,0 days 00:11:04.743195,1,0.418329,3,0,0.003855,0.002809,19,0.050181,4,1,1.247711,1,0.010489,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,218,0.914625,COMPLETE,0 days 01:08:29.542138,1,0.698261,3,4,0.119732,0.001299,7,0.243425,2,1,1.450697,0,0.027687,2
257,257,0.915667,COMPLETE,0 days 01:08:55.483219,1,0.691018,3,4,0.131799,0.001193,7,0.230646,2,1,1.108544,0,0.025188,2
348,348,0.917792,COMPLETE,0 days 01:19:35.522959,1,0.695326,0,4,0.134206,0.002691,7,0.222224,2,2,1.394869,0,0.090252,0
339,339,0.921000,COMPLETE,0 days 01:12:12.271855,1,0.673334,3,4,0.125110,0.001381,7,0.242437,2,1,1.458563,0,0.020457,2


## Optimization history

In [5]:
from optuna.visualization import plot_optimization_history
plot_optimization_history(study)

## Parameter importance

Uses fANOVA to estimate how much variance in the objective each parameter explains.

In [6]:
from optuna.visualization import plot_param_importances
plot_param_importances(study)

## Parallel coordinates

Each line is a trial. Colour encodes the objective value.

In [7]:
from optuna.visualization import plot_parallel_coordinate
plot_parallel_coordinate(study)

## Slice plot

Objective value vs. each parameter individually.

In [8]:
from optuna.visualization import plot_slice
plot_slice(study)

## Hyperparameter contour

Pairwise interaction between parameters.

In [9]:
from optuna.visualization import plot_contour
best = study.best_trials[0] if multi_obj else study.best_trial
params = list(best.params.keys())[:2]
plot_contour(study, params=params)

## Pareto front (multi-objective only)

In [10]:
if multi_obj:
    from optuna.visualization import plot_pareto_front
    plot_pareto_front(study)
else:
    print("Single-objective study — no Pareto front to plot.")

Single-objective study — no Pareto front to plot.
